In [4]:
!ls


'kaggle (1).json'   sample_data


In [6]:
!mkdir -p ~/.kaggle
!cp 'kaggle (1).json' ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json



In [7]:
!kaggle datasets list


ref                                                             title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
shahzadi786/world-smartphone-market-2025                        World Smartphone Market 2025                             17795  2025-11-09 04:52:42.650000            756         30  1.0              
ahmeduzaki/global-earthquake-tsunami-risk-assessment-dataset    Global Earthquake-Tsunami Risk Assessment Dataset        16151  2025-10-01 16:35:53.273000          19375        655  1.0              
ahmadrazakashif/bmw-worldwide-sales-records-20102024            BMW Worldwide Sales Records (2010–2024)                 853348  2025-09-20 14:39:45.280000          23613        463  1.0              


In [8]:
!kaggle datasets download -d lava18/google-play-store-apps


Dataset URL: https://www.kaggle.com/datasets/lava18/google-play-store-apps
License(s): CC-BY-SA-4.0
  0% 0.00/1.94M [00:00<?, ?B/s]
100% 1.94M/1.94M [00:00<00:00, 775MB/s]


In [9]:
!unzip google-play-store-apps.zip


Archive:  google-play-store-apps.zip
  inflating: googleplaystore.csv     
  inflating: googleplaystore_user_reviews.csv  
  inflating: license.txt             


In [10]:
import pandas as pd

df = pd.read_csv("googleplaystore.csv")
df.head()


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [21]:

df = df.drop_duplicates()

df['Installs'] = df['Installs'].astype(str)
df['Installs'] = df['Installs'].str.replace('+','', regex=False)
df['Installs'] = df['Installs'].str.replace(',','', regex=False)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

df['Price'] = df['Price'].astype(str).str.replace('$','', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

df['Size'] = df['Size'].astype(str)
df['Size'] = df['Size'].replace('Varies with device', None)
df['Size'] = df['Size'].str.replace('M','', regex=False)
df['Size'] = pd.to_numeric(df['Size'], errors='coerce')

df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

df["desc_len"] = df["Content Rating"].astype(str).apply(len)

df["desc_words"] = df["Content Rating"].astype(str).apply(lambda x: len(x.split()))

df["title_free"] = df["App"].str.lower().str.contains("free").astype(int)

scam_words = ["reward", "bonus", "prize", "claim", "win", "earn"]
df["suspicious_text"] = df["Content Rating"].str.lower().apply(
    lambda x: any(word in x for word in scam_words)
).astype(int)

df["is_free"] = df["Type"].apply(lambda x: 1 if x=="Free" else 0)

df["Category"] = df["Category"].astype("category")
df["category_code"] = df["Category"].cat.codes

df = df.dropna(subset=['Rating', 'Installs', 'Reviews', 'Size'])


In [22]:
df['is_fake'] = 0

df.loc[df['Rating'] < 3.0, 'is_fake'] = 1

df.loc[df['Installs'] < 10000, 'is_fake'] = 1

X = df[['Reviews',
        'Size',
        'desc_len',
        'desc_words',
        'title_free',
        'suspicious_text',
        'is_free',
        'category_code']]

y = df['is_fake']


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [24]:
#MODEL 1
from sklearn.linear_model import LogisticRegression

log = LogisticRegression()
log.fit(X_train, y_train)

log_acc = log.score(X_test, y_test)
print("Logistic Regression Accuracy:", log_acc)


Logistic Regression Accuracy: 0.8940027894002789


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [29]:
#MODEL 2
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

dt_acc = dt.score(X_test, y_test)
print("Decision Tree Accuracy:", dt_acc)


Decision Tree Accuracy: 0.899581589958159


In [26]:
#model 3
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

rf_acc = rf.score(X_test, y_test)
print("Random Forest Accuracy:", rf_acc)


Random Forest Accuracy: 0.9330543933054394


In [27]:
#model 4
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)

svm_acc = svm.score(X_test, y_test)
print("SVM Accuracy:", svm_acc)


SVM Accuracy: 0.7489539748953975


In [28]:
print("\n=== Model Accuracy Comparison ===")
print("Logistic Regression:", log_acc)
print("Decision Tree:", dt_acc)
print("Random Forest:", rf_acc)
print("SVM:", svm_acc)



=== Model Accuracy Comparison ===
Logistic Regression: 0.8940027894002789
Decision Tree: 0.900278940027894
Random Forest: 0.9330543933054394
SVM: 0.7489539748953975
